# Полные предложения с подвохами на больших моделях

Прогон `scripts/sentences_tricks.py` (протокол записан в начале файла и закоммичен до прогона) на моделях больше и новее: GPT-2 XL, Qwen3-8B, Qwen3-14B и Qwen2.5-7B. Интерпретатор здесь не обучается: его веса лежат в репозитории (`results/sentences_tricks_interpreter.pt`), поэтому все модели читаются одними и теми же весами.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → GPU (T4). Затем ячейки по очереди. Каждый прогон пишет свой файл и не перезаписывает существующий.

In [ ]:
!nvidia-smi

Код скачивается прямо из репозитория, вместе с весами интерпретатора.

In [ ]:
!rm -rf whybox && git clone -q https://github.com/knbww/whybox.git
%cd whybox

In [ ]:
!pip install -q "transformers>=4.56" pyyaml pytest bitsandbytes accelerate

Проверка, что предложения грамматичны до и после каждого подавления.

In [ ]:
!python -m pytest -q tests/test_tricks.py

## GPT-2 XL, 1,5 млрд параметров (float32)

Самая большая GPT-2. Около 10 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models openai-community/gpt2-xl \
    --device cuda --dtype float32 \
    --out results/sentences_tricks_colab_gpt2xl.json

## Qwen3-8B (4 бита)

Новое поколение Qwen. С градиентами в T4 помещается только в 4-битном виде (поправка 1 к протоколу); хватает ли точности, покажет ошибка полноты в выводе. Скачивание около 16 ГБ, всего около 20 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-8B \
    --device cuda --load-in-4bit \
    --out results/sentences_tricks_colab_qwen3_8b.json

## Qwen3-14B (4 бита)

Самая большая модель, которая может поместиться в бесплатную T4. Скачивание около 30 ГБ, всего около 30–40 минут. Если не хватит памяти, ячейка упадёт с ошибкой CUDA out of memory, а результаты предыдущих ячеек останутся.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-14B \
    --device cuda --load-in-4bit \
    --out results/sentences_tricks_colab_qwen3_14b.json

## По желанию: Qwen2.5-7B (4 бита)

То же семейство, что Qwen2.5 0,5–3B из прошлых прогонов, только больше. Около 20 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen2.5-7B-Instruct \
    --device cuda --load-in-4bit \
    --out results/sentences_tricks_colab_qwen25_7b.json

## Итог и скачивание результатов

In [ ]:
import glob, json
from google.colab import files
new = ['gpt2xl', 'qwen3_8b', 'qwen3_14b', 'qwen25_7b']
for f in sorted(glob.glob('results/sentences_tricks_colab_*.json')):
    if not any(n in f for n in new):
        continue
    d = json.load(open(f))
    for model, by_t in d['results'].items():
        for t, r in by_t.items():
            acc = {k: round(v, 3) for k, v in r['arms']['interpreter'].items() if v == v}
            print(f"{model:30s} {t:8s} {r['read_out']:26s} interpreter {acc}  "
                  f"live {r['arms']['live']['ALL']:.3f}  completeness {r['completeness_residual'][0]:.4f}")
    files.download(f)